# Módulo 4 — Aterrizaje Propulsado (Landing)
**Programa:** Inspira STEM 2026  
**Instructores:** Oscar Tejada y Patricia Ortíz

---
## La Fase Terminal y Control Vectorial
A 1.8 kilómetros de la superficie, la densidad atmosférica resulta insuficiente y la influencia del paracaídas concluye. El sistema se separa y entra en caída libre, confiando su supervivencia a la etapa de descenso (la revolucionaria maniobra *Sky Crane* implementada en Curiosity).

Este vehículo dependió de 8 motores aceleradores regulables de hidracina (Mars Lander Engines) que suman 24.8 kN, generando empuje retrogrado para reducir la velocidad desde aproximadamente 80 m/s hasta apenas 0.75 m/s, manteniendo un vuelo estacionario (hover) milimétrico para descender el rover mediante un umbilical de nylon.

La etapa llega a esta fase con 2,400 kg: 2,010 kg de estructura, motores y rover, más 390 kg de hidracina utilizable. El impulso específico de un monopropelente de hidracina ronda los 210 s, muy por debajo de los 300 s de un bipropelente. El requisito de 301 m/s no es la velocidad de caída: incluye los 161 m/s que se pierden sosteniendo el peso del vehículo contra la gravedad mientras frena.

La viabilidad termodinámica y orbital de esta maniobra se consolida evaluando tres balances paramétricos fundamentales.

---

### Paso 1: Configuración de la Computadora de Vuelo
Ejecuta la celda inferior para compilar el módulo de empuje de la computadora de a bordo. Recuerda registrar con precisión la gravedad local y el requisito de cambio de velocidad ($\Delta v$) heredado de la pérdida de eficiencia del paracaídas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

plt.rcParams.update({'figure.dpi': 100, 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3, 'lines.linewidth': 2})

# ==========================================
# ⚙️ DATOS DEL ENTORNO Y REQUISITO CINEMÁTICO
# ==========================================
g_planeta = 3.71        # Gravedad local [m/s^2]
dv_requerido = 301.0    # Delta-v de la fase propulsada [m/s]

def calc_landing(m_seca, m_prop, isp, empuje):
    m_tot = m_seca + m_prop
    return {
        "m_tot": m_tot,
        "peso_kn": (m_tot * g_planeta) / 1000,
        "empuje_kn": empuje / 1000,
        "twr": empuje / (m_tot * g_planeta),
        "dv_max": isp * 9.81 * np.log(m_tot / m_seca),
        "tb": (m_prop * isp * 9.81) / empuje
    }
print("Sistemas de retropropulsión en línea.")

---
### Análisis 1: Relación Empuje-Peso (TWR)
La autoridad de control vertical absoluta está dictaminada por la relación *Thrust-to-Weight Ratio* (TWR). Este indicador contrasta el **Empuje total ($\mathbf{F}$)** generado por el bloque motriz frente a la atracción gravitatoria sobre la masa del sistema:
$$TWR = \frac{\mathbf{F}}{m_{total} \cdot g_{local}}$$

La exploración de este parámetro, manipulando el Empuje ($\mathbf{F}$), permite comprobar visualmente que la maniobra de frenado requiere estrictamente que el vector de fuerza ascendente (empuje operativo) supere el componente vectorial de descenso (peso local), logrando un $TWR > 1$.

**Qué esperar al mover el deslizador.** El TWR es una condición de sí o no: por debajo de 1 el vehículo cae con los motores encendidos.

| Si el empuje sube | Si el empuje baja |
|---|---|
| El TWR sube en proporción directa | El TWR baja |
| Más autoridad para frenar y para sostenerse | Menos margen sobre el peso |
| El tiempo de encendido del Análisis 3 baja | El tiempo de encendido sube |

Fíjate en la gráfica izquierda: la misma nave con el mismo motor da un TWR distinto en cada planeta, porque el denominador lo pone la gravedad local. Y fíjate sobre todo en la última fila de la tabla: el empuje aparece en el numerador de este análisis y en el denominador del Análisis 3. Subirlo arregla una cosa y rompe la otra.

In [ ]:
def interact_empuje(empuje_kn):
    d = calc_landing(m_seca=2010.0, m_prop=390.0, isp=210.0, empuje=empuje_kn*1000)
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
    
    gravedades = np.linspace(1.0, 20.0, 50)
    twrs = (empuje_kn * 1000) / (d["m_tot"] * gravedades)
    axs[0].plot(gravedades, twrs, color="#c0392b")
    axs[0].axhline(1.0, color="k", linestyle="--", label="Límite Operacional Crítico (TWR = 1)")
    axs[0].axvline(g_planeta, color="#7f8c8d", linestyle=":", label=f"Gravedad local ({g_planeta} m/s²)")
    axs[0].set(title="Comportamiento TWR frente a Gravedades Planetarias", xlabel="Gravedad Local [m/s²]", ylabel="Relación Empuje/Peso")
    axs[0].legend()
    
    axs[1].bar(["Peso Sistémico"], [d["peso_kn"]], color="#c0392b")
    axs[1].bar(["Empuje de Reacción"], [d["empuje_kn"]], color="#27ae60")
    axs[1].set(title=f"Balance de Fuerzas Verticales (TWR Actual: {d['twr']:.2f})", ylabel="Fuerza Neta [kN]")
    plt.show()

interact(interact_empuje, empuje_kn=widgets.FloatSlider(value=24.8, min=5.0, max=80.0, step=0.2, description='Empuje [kN]:'));

---
### Análisis 2: Presupuesto Cinemático y Ecuación de Tsiolkovsky
El monto total de cambio de velocidad ($\,\Delta v\,$) que la configuración propulsiva puede brindar se rige incondicionalmente por la Ecuación del Cohete de Tsiolkovsky.

Esta ley física subraya el impacto crítico del rendimiento termodinámico de la ignición, cuantificado a través del **Impulso Específico ($\mathbf{I_{sp}}$)**:
$$\Delta v = \mathbf{I_{sp}} \cdot g_0 \cdot \ln\left(\frac{m_{total}}{m_{seca}}\right) \quad [\text{m/s}]$$

Iterando sobre el Impulso Específico ($I_{sp}$), se simula la transición entre distintas arquitecturas químicas de propelentes (ej. monopropelentes de hidracina frente a sistemas bipropelentes más complejos) para garantizar que la capacidad $\Delta v$ de la plataforma cumpla con el requerimiento de la misión.

**Qué esperar al mover el deslizador.** El $I_{sp}$ multiplica por fuera del logaritmo y la masa entra por dentro. No es lo mismo.

| Si el $I_{sp}$ sube | Si se carga más propelente |
|---|---|
| El $\Delta v$ sube en proporción directa | El $\Delta v$ sube cada vez menos |
| No cambia la masa del vehículo | La masa total sube y castiga las fases 1 y 2 |
| Exige otra química de propelente | Solo exige tanques más grandes |

Mira la curva de la izquierda: para duplicar el $\Delta v$ hay que elevar al cuadrado la fracción de masa. Ese logaritmo es la mala noticia de toda la astronáutica, y explica por qué se invierte tanto en mejorar el motor y no simplemente en agrandar el tanque.

In [ ]:
def interact_isp(isp_segundos):
    d = calc_landing(m_seca=2010.0, m_prop=390.0, isp=isp_segundos, empuje=24800.0)
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
    
    fracciones = np.linspace(1.1, 2.5, 50)
    dvs = isp_segundos * 9.81 * np.log(fracciones)
    axs[0].plot(fracciones, dvs, color="#2980b9")
    axs[0].set(title="Curva Potencial de Tsiolkovsky", xlabel="Fracción de Masa (Húmeda / Seca)", ylabel="$\Delta v$ Alcanzable [m/s]")
    
    axs[1].barh(["Capacidad de la Etapa"], [d["dv_max"]], color="#2980b9")
    axs[1].axvline(dv_requerido, color="#e67e22", linestyle="--", lw=3, label=f"Requisito Mínimo ({dv_requerido} m/s)")
    axs[1].set(title="Presupuesto Cinemático Consolidado", xlabel="Cambio de Velocidad $\Delta v$ [m/s]")
    axs[1].legend()
    plt.show()

interact(interact_isp, isp_segundos=widgets.FloatSlider(value=210.0, min=150.0, max=450.0, step=5.0, description='Eficiencia (Isp):'));

---
### Análisis 3: Restricción Temporal de Vuelo Estacionario
La maniobra crítica de separación y descenso del rover con poleas y cables demanda mantener un estado de *hover* preciso (aceleración neta de 0 m/s²) durante los últimos segundos. Esta ventana de ignición prolongada está coartada por el volumen de **Masa del Propelente ($\mathbf{m_p}$)** en los depósitos de hidracina:
$$t_b = \frac{\mathbf{m_p} \cdot I_{sp} \cdot g_0}{F} \quad [\text{s}]$$

La evaluación de la capacidad de los tanques ($m_p$) revela la fragilidad temporal de la etapa terminal, evidenciando la necesidad de conservar márgenes de fluido operativo hasta el instante del contacto superficial.

**Qué esperar al mover el deslizador.** El propelente es el único parámetro que ayuda y estorba a la vez.

| Si el propelente sube | Consecuencia en otra parte |
|---|---|
| El tiempo de encendido sube en proporción directa | La masa total sube |
| El $\Delta v$ del Análisis 2 sube | El TWR del Análisis 1 baja |
| — | El coeficiente balístico de la fase 1 empeora |

Y en la gráfica de la izquierda tienes la otra mitad del problema: con el propelente fijo, el tiempo de encendido cae en proporción inversa al empuje. El motor más potente es también el que antes se queda seco. Las preguntas 1 y 2 de grupo salen las dos de esta tensión.

In [ ]:
def interact_tanque(masa_tanque):
    d = calc_landing(m_seca=2010.0, m_prop=masa_tanque, isp=210.0, empuje=24800.0)
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
    
    empujes = np.linspace(5, 80, 50)
    t_burns = (masa_tanque * 210.0 * 9.81) / (empujes * 1000)
    axs[0].plot(empujes, t_burns, color="#8e44ad")
    axs[0].set(title="Tasa de Agotamiento vs Fuerza Propulsiva", xlabel="Empuje Operativo [kN]", ylabel="Duración Máxima [s]")
    
    axs[1].bar(["Margen Restante"], [d["tb"]], color="#8e44ad")
    axs[1].axhline(30.0, color="#2c3e50", linestyle="--", lw=2, label="Ventana de Descenso (30s)")
    axs[1].set(title="Límite Temporal del Aterrizaje", ylabel="Tiempo [s]")
    axs[1].legend()
    plt.show()

interact(interact_tanque, masa_tanque=widgets.FloatSlider(value=390.0, min=50.0, max=1200.0, step=10.0, description='Propelente [kg]:'));

### Análisis de Grupo
Reúnanse y debatan las exigencias y dependencias paramétricas del aterrizaje propulsado:

1. **Configuración de Motores (Misión Real):** El vehículo de descenso de Curiosity utilizó 8 motores aceleradores MLE (Mars Lander Engines) distribuidos alrededor de la cápsula. Si por un exceso de "seguridad" los ingenieros hubieran ordenado que operaran a su empuje máximo estructural para frenar lo antes posible, ¿qué crisis inmediata se habría detonado según la matemática del Análisis 3?
2. **El Reloj de la Grúa Aérea:** Con los valores reales, el Análisis 3 da unos 32 segundos de encendido y la maniobra de *Sky Crane* exigía entre 21 y 25 segundos de vuelo estacionario. Pero antes de eso hay que frenar de 80 m/s a cero, y los números no cierran salvo que los motores trabajen regulados muy por debajo de su empuje máximo. Expliquen cómo la regulación resuelve la contradicción y por qué el dato de 24.8 kN es un techo y no un consumo.
3. **El Escape de la Grúa (Misión Real):** En agosto de 2012, una vez que las ruedas del Curiosity tocaron el cráter Gale y los umbilicales pirotécnicos se cortaron, los motores de la grúa no se apagaron. Su sistema de vuelo ordenó una maniobra de *Flyaway* (Escape), acelerando a máxima potencia hacia arriba e inclinándose para alejarse del rover e impactar intencionalmente a 650 metros de distancia. Observando sus gráficas del Análisis 2 y 3, ¿qué requerimiento exigía esta maniobra de escape obligatorio al diseño original de los depósitos de hidracina, y por qué la NASA nunca dimensiona el combustible para llegar exactamente a cero en el instante de contacto?